# Text Generation via Prompt Lookup Decoding using OpenVINO™

As model sizes grow, Generative AI implementations require significant inference resources. This not only increases the cost per generation from a prompt, but also increases the power consumption used to serve such requests.

Inference optimizations for text generation are essential for reducing costs and power consumption. When optimizing the inference process, the amount of time and energy required to generate text can be significantly reduced. This can lead to cost savings in terms of hardware and software, as well as reduced power consumption. Additionally, inference optimizations can help improve the accuracy of text generation as well as the speed at which it can be generated. This can lead to an improved user experience and increased efficiency in text-generation tasks. In summary, inference optimizations for text generation are essential to reduce costs and power consumption, while also improving the accuracy and speed of text generation.

[Prompt Lookup decoding](https://github.com/apoorvumang/prompt-lookup-decoding) is [assisted-generation](https://huggingface.co/blog/assisted-generation#understanding-text-generation-latency) technique, that allows to speed up token generation, where the draft model is replaced with simple string matching the prompt to generate candidate token sequences. 

Prompt Lookup decoding works the following way. Input defines as all the tokens till the current generation step (input_ids). It then tries to match last few tokens to somewhere earlier in the prompt. If found, it returns the next-k token continuation as `candidate input ids` or `candidate sequence`.

![](https://blog.vllm.ai/assets/figures/spec-decode/figure3.png)

This method highly effective for input grounded generation (summarization, document QA, multi-turn chat, code editing), where there is high n-gram overlap between LLM input (prompt) and LLM output. This could be entity names, phrases, or code chunks that the LLM directly copies from the input while generating the output. Prompt lookup exploits this pattern to speed up autoregressive decoding in LLMs. This results in significant speedups with no effect on output quality.

In this tutorial we consider how to apply [Prompt Lookup decoding with OpenVINO GenAI](https://medium.com/openvino-toolkit/enhancing-llm-inference-with-prompt-lookup-decoding-and-openvino-genai-e15b69aeaeab).

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Prepare models](#Prepare-models)
    - [Select inference device](#Select-inference-device)
- [Run target model without prompt lookup decoding](#Run-target-model-without-prompt-lookup-decoding)
- [Run Prompt Lookup decoding pipeline](#Run-Prompt-Lookup-decoding-pipeline)
- [Evaluate Prompt Lookup Decoding on multiple examples](#Evaluate-Prompt-Lookup-Decoding-on-multiple-examples)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/prompt-lookup-decoding/prompt-lookup-decoding.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)


First, we should install the [OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) for running model inference.

![](https://media.githubusercontent.com/media/openvinotoolkit/openvino.genai/refs/heads/master/src/docs/openvino_genai.svg)

[OpenVINO™ GenAI](https://github.com/openvinotoolkit/openvino.genai) is a library of the most popular Generative AI model pipelines, optimized execution methods, and samples that run on top of highly performant [OpenVINO Runtime](https://github.com/openvinotoolkit/openvino).

This library is friendly to PC and laptop execution, and optimized for resource consumption. It requires no external dependencies to run generative models as it already includes all the core functionality (e.g. tokenization via openvino-tokenizers).


In [ ]:
%pip install --pre -U openvino-genai --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly huggingface_hub datasets

## Prepare models
[back to top ⬆️](#Table-of-contents:)

As example, we will use already converted LLMs from [OpenVINO collection](https://huggingface.co/collections/OpenVINO/llm-6687aaa2abca3bbcec71a9bd) [open_llama_7b_v2](https://huggingface.co/OpenVINO/TinyLlama-1.1B-Chat-v1.0-int4-ov).

In case, if you want run own models, you should convert them using [Hugging Face Optimum](https://huggingface.co/docs/optimum/intel/openvino/export) library accelerated by OpenVINO integration. More details about model preparation can be found in [OpenVINO LLM inference guide](https://docs.openvino.ai/2024/learn-openvino/llm_inference_guide/llm-inference-native-ov.html#convert-hugging-face-tokenizer-and-model-to-openvino-ir-format)

In [18]:
from pathlib import Path
import huggingface_hub as hf_hub

model_id = "OpenVINO/TinyLlama-1.1B-Chat-v1.0-int4-ov"

model_path = Path(model_id.split("/")[-1])

if not model_path.exists():
    hf_hub.snapshot_download(model_id, local_dir=model_path)

### Select inference device
[back to top ⬆️](#Table-of-contents:)


Select the device from dropdown list for running inference using OpenVINO.
> **Note**: For achieving maximal performance, we recommend to use GPU as target device if it is available.

In [19]:
import requests

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU", "AUTO"])

device

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("prompt-lookup-decoding.ipynb")

## Run target model without prompt lookup decoding
[back to top ⬆️](#Table-of-contents:)

OpenVINO GenAI provides easy-to-use API for running text generation. Firstly we will create pipeline with `LLMPipeline`. `LLMPipeline` is the main object used for decoding. You can construct it straight away from the folder with the converted model. It will automatically load the `main model`, `tokenizer`, `detokenizer` and default `generation configuration`. 
After that we will configure parameters for decoding. 
Then we just run `generate` method and get the output in text format. We do not need to encode input prompt according to model expected template or write post-processing code for logits decoder, it will be done easily with LLMPipeline. 

To obtain intermediate generation results without waiting until when generation is finished, we will write streamer function.

In [20]:
import openvino_genai as ov_genai
import time

pipe = ov_genai.LLMPipeline(model_path, device.value)

config = ov_genai.GenerationConfig()
config.max_new_tokens = 330
prompt = '''<s>
def prime_fib(n: int):
    """
    prime_fib returns n-th number that is a Fibonacci number and it's also prime.
    >>> prime_fib(1)
    2
    >>> prime_fib(2)
    3
    >>> prime_fib(3)
    5
    >>> prime_fib(4)
    13
    >>> prime_fib(5)
    89
    """'''


def streamer(subword):
    print(subword, end="", flush=True)
    # Return flag corresponds whether generation should be stopped.
    # False means continue generation.
    return False


start_time = time.perf_counter()
pipe.generate(prompt, config, streamer=streamer)
end_time = time.perf_counter()

The `prime_fib` function in Python takes an integer `n` and returns the n-th Fibonacci number and also checks if it is prime.

The function first checks if `n` is a prime number. If it is not, it returns `None`.

If `n` is a prime number, the function first checks if `n` is a multiple of 2. If it is not, it returns `None`.

If `n` is a multiple of 2, the function first checks if `n` is a multiple of 4. If it is not, it returns `None`.

If `n` is a multiple of 4, the function first checks if `n` is a multiple of 8. If it is not, it returns `None`.

If `n` is a multiple of 8, the function first checks if `n` is a multiple of 16. If it is not, it returns `None`.

If `n` is a multiple of 16, the function first checks if `n` is a multiple of 32. If it is not, it returns `None`.

If `n` is a multiple of 32, the function first checks if `n` is a multiple of 64. If it is not, it returns `None`.

If `n` is a multiple of 64, the function first checks if `n` is a multiple of 128. If it is not, it

In [21]:
import gc

print(f"Generation time: {end_time - start_time:.2f}s")
del pipe
gc.collect()

Generation time: 6.56s


0

## Run Prompt Lookup decoding pipeline
[back to top ⬆️](#Table-of-contents:)

To enable prompt lookup decoding in `LLMPipeline,` we should setup parameter `prompt_lookup` to `True`. Additionally we can provide `SchedulerConfig` for resource management. 
We also need to specify two parameters via generation config: `num_assistant_tokens` is the candidate sequence length to return per iteration and `max_ngram_size` is the maximum ngram to use when looking for matches in the prompt.

In [22]:
pipe = ov_genai.LLMPipeline(model_path, device.value, prompt_lookup=True)

config = ov_genai.GenerationConfig()
config.max_new_tokens = 330
config.num_assistant_tokens = 5
config.max_ngram_size = 3
start_time = time.perf_counter()
result = pipe.generate(prompt, config, streamer=streamer)
end_time = time.perf_counter()

The `prime_fib` function in Python takes an integer `n` and returns the n-th Fibonacci number and also checks if it is prime.

The function first checks if `n` is a prime number. If it is not, it returns `None`.

If `n` is a prime number, the function first checks if `n` is divisible by any of the Fibonacci numbers in the range `[1, n-1]`. If any of these numbers is divisible by `n`, the function returns `None`.

Otherwise, the function calculates the Fibonacci numbers `fib_1` and `fib_2` such that `fib_1 + fib_2 = n`.

The function then checks if `fib_1` and `fib_2` are not equal to `n`. If they are, the function returns `None`.

Finally, the function checks if `fib_1` and `fib_2` are not equal to `n-1`. If they are, the function returns `None`.

Therefore, the `prime_fib` function returns the n-th Fibonacci number and also checks if it is prime.

In [23]:
print(f"Generation time: {end_time - start_time:.2f}s")

Generation time: 4.03s


## Evaluate Prompt Lookup Decoding on multiple examples
[back to top ⬆️](#Table-of-contents:)

Configure the data type and the number of examples to run:

In [24]:
num_samples_to_select = 50

import ipywidgets as widgets

data_options = ["Code", "Text"]
data_type = widgets.Dropdown(
    options=data_options,
    value=data_options[0],
    description="Data type:",
    disabled=False,
)
data_type

Dropdown(description='Data type:', options=('Code', 'Text'), value='Code')

Load the dataset and prepare the prompts:

In [25]:
from datasets import load_dataset

print("loading dataset...")

if data_type.value == "Code":
    ds = load_dataset("openai_humaneval", split="test")
    prompts = ds["prompt"]
    prompts = ["<s>" + prompts[i] for i in range(num_samples_to_select)]
else:
    ds = load_dataset("abisee/cnn_dailymail", "3.0.0", split="test")
    prompts = ds["article"]
    prompts = [
        "<|user|> ###\nArticle: " + prompts[i] + "\n\nSummarize the above article in 5 sentence.\n<|end|><|assistant|>" for i in range(num_samples_to_select)
    ]
print("Done")

loading dataset...
Done


Run auto-regressive generation and get total runtime per example:

In [26]:
import openvino_genai as ov_genai
import time
from tqdm import tqdm

print("Running Auto-Regressive generation...")
pipe = ov_genai.LLMPipeline(model_path, device.value)

config = ov_genai.GenerationConfig()
config.max_new_tokens = 330

times_auto_regressive = []
for prompt in tqdm(prompts):
    start_time = time.perf_counter()
    result = pipe.generate(prompt, config)
    end_time = time.perf_counter()
    times_auto_regressive.append(end_time - start_time)
print("Done")

import gc

del pipe
gc.collect()

Running Auto-Regressive generation...


100%|██████████| 50/50 [04:01<00:00,  4.82s/it]

Done


25

Now run generation with Prompt Lookup decoding:

In [27]:
pipe = ov_genai.LLMPipeline(model_path, device.value, prompt_lookup=True)

config = ov_genai.GenerationConfig()
config.max_new_tokens = 330
config.num_assistant_tokens = 5
config.max_ngram_size = 3


times_prompt_lookup = []
print("Running Prompt Lookup Decoding generation...")
for prompt in tqdm(prompts):
    start_time = time.perf_counter()
    result = pipe.generate(prompt, config)
    end_time = time.perf_counter()
    times_prompt_lookup.append((end_time - start_time))
print("Done")

Running Prompt Lookup Decoding generation...


100%|██████████| 50/50 [03:03<00:00,  3.68s/it]

Done


Now let's calculate the speedup:

In [28]:
avg_speedup = sum([x / y for x, y in zip(times_auto_regressive, times_prompt_lookup)]) / len(prompts)
print(f"average speedup: {avg_speedup:.2f}")

average speedup: 1.37
